# MASI Kaggle Medium Smoke Test

This notebook runs the bounded Kaggle medium workflow end-to-end for MASI.

Expected Kaggle inputs:

- a dataset mounted at `/kaggle/input/masi-amazon-csj-raw`
- files `Clothing_Shoes_and_Jewelry.jsonl` and `meta_Clothing_Shoes_and_Jewelry.jsonl`
- optionally, a second resume dataset copied into `/kaggle/input/<resume-slug>`

Outputs are written under `/kaggle/working/masi_artifacts` by default and can be exported at the end of the notebook.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

INPUT_ROOT = Path("/kaggle/input")
RAW_DATASET_SLUG = "masi-amazon-csj-raw"
RAW_DATASET_DIR = INPUT_ROOT / RAW_DATASET_SLUG
RESUME_DATASET_SLUG = None
RESUME_DATASET_DIR = INPUT_ROOT / RESUME_DATASET_SLUG if RESUME_DATASET_SLUG else None

REPO_DIR = Path("/kaggle/working/MASI")
STORAGE_ROOT = Path("/kaggle/working/masi_artifacts")
RUN_NAME = "amazon_csj_medium_kaggle_train"
RUN_ROOT = STORAGE_ROOT / "outputs" / RUN_NAME
CACHE_ROOT = STORAGE_ROOT / "data" / "processed" / "amazon_csj_medium_kaggle"

RAW_REVIEW_PATH = RAW_DATASET_DIR / "Clothing_Shoes_and_Jewelry.jsonl"
RAW_METADATA_PATH = RAW_DATASET_DIR / "meta_Clothing_Shoes_and_Jewelry.jsonl"

if not RAW_DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Attach the Kaggle Dataset '{RAW_DATASET_SLUG}' so the notebook can read raw CSJ files from {RAW_DATASET_DIR}."
    )

for required_path in [RAW_REVIEW_PATH, RAW_METADATA_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required raw input file: {required_path}")

candidates = []
for pyproject_path in INPUT_ROOT.glob("**/pyproject.toml"):
    candidate = pyproject_path.parent
    if (candidate / "scripts" / "train_masi.py").exists() and (candidate / "src" / "masi").exists():
        candidates.append(candidate)

if not candidates:
    raise FileNotFoundError("Could not find the uploaded MASI repository under /kaggle/input.")

repo_source = sorted(candidates, key=lambda path: (len(path.parts), str(path)))[0]
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(repo_source, REPO_DIR)
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
print(f"Copied repo from {repo_source} to {REPO_DIR}")
print(f"Using raw dataset from {RAW_DATASET_DIR}")
print(f"Working storage root: {STORAGE_ROOT}")
print(f"Run root: {RUN_ROOT}")
if RESUME_DATASET_DIR is not None:
    print(f"Will restore previous artifacts from {RESUME_DATASET_DIR}")

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[recommender]"], check=True)


In [ ]:
from pathlib import Path
import json
import shutil

if RESUME_DATASET_DIR is None:
    print("No resume dataset attached; starting from a fresh working directory.")
else:
    if not RESUME_DATASET_DIR.exists():
        raise FileNotFoundError(f"Resume dataset path does not exist: {RESUME_DATASET_DIR}")
    for source_path in sorted(RESUME_DATASET_DIR.iterdir()):
        destination_path = STORAGE_ROOT / source_path.name
        if destination_path.exists():
            if destination_path.is_dir():
                shutil.rmtree(destination_path)
            else:
                destination_path.unlink()
        if source_path.is_dir():
            shutil.copytree(source_path, destination_path)
        else:
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_path)
    print(f"Restored prior artifacts from {RESUME_DATASET_DIR} into {STORAGE_ROOT}")

config_path = REPO_DIR / "configs" / "masi_train_csj_medium_kaggle.json"
config = json.loads(config_path.read_text())
print(json.dumps(config, indent=2))


## Optional prefetch step

This warms the image cache and writes `image_download_manifest.json` before the full training run. It is useful on Kaggle because image downloading can be the most failure-prone stage.

In [ ]:
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run(
    [
        sys.executable,
        "scripts/download_amazon_csj_images.py",
        "--config",
        "configs/masi_train_csj_medium_kaggle.json",
        "--storage-root",
        str(STORAGE_ROOT),
        "--workers",
        "16",
        "--retries",
        "3",
        "--resume",
    ],
    check=True,
    env={**os.environ, "PYTHONPATH": "src"},
)


## Run the medium smoke test

This executes the full bounded MASI pipeline:

1. bounded dataset selection and metadata slicing
2. CLIP text/image encoding
3. Phase 1 behavior-aware alignment
4. Phase 2 text and vision RQ-VAE training
5. fused semantic ID export
6. Phase 3 MLM pretraining, autoregressive fine-tuning, and evaluation

Periodic step checkpoints are saved every `25` optimizer steps per stage by the Kaggle medium config.

In [ ]:
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run(
    [
        sys.executable,
        "scripts/train_masi.py",
        "--config",
        "configs/masi_train_csj_medium_kaggle.json",
        "--storage-root",
        str(STORAGE_ROOT),
    ],
    check=True,
    env={**os.environ, "PYTHONPATH": "src"},
)


In [ ]:
from pathlib import Path
import json

summary_paths = {
    "image_download_manifest": RUN_ROOT / "image_download_manifest.json",
    "run_manifest": RUN_ROOT / "run_manifest.json",
    "token_summary": RUN_ROOT / "phase12_tokens" / "masi_token_summary.json",
    "experiment_summary": RUN_ROOT / "phase3_experiment" / "experiment_summary.json",
}

for label, path in summary_paths.items():
    print(f"\n=== {label}: {path} ===")
    if not path.exists():
        print("missing")
        continue
    payload = json.loads(path.read_text())
    print(json.dumps(payload, indent=2)[:4000])

phase12_checkpoint_root = RUN_ROOT / "checkpoints" / "phase12_tokens"
phase3_checkpoint_root = RUN_ROOT / "checkpoints" / "phase3_experiment"

print("\nPhase 1/2 checkpoints:")
for path in sorted(phase12_checkpoint_root.rglob("*")):
    print(path)

print("\nPhase 3 checkpoints:")
for path in sorted(phase3_checkpoint_root.rglob("*")):
    print(path)


In [ ]:
from pathlib import Path
import shutil
import subprocess

EXPORT_ROOT = Path("/kaggle/working/MASI_medium_smoke_export")
ZIP_PATH = Path("/kaggle/working/MASI_medium_smoke_export.zip")

if not RUN_ROOT.exists():
    raise FileNotFoundError(f"Run output directory does not exist yet: {RUN_ROOT}")

if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

export_run_root = EXPORT_ROOT / "outputs" / RUN_ROOT.name
export_run_root.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_ROOT, export_run_root)

if CACHE_ROOT.exists():
    export_cache_root = EXPORT_ROOT / "data" / "processed" / CACHE_ROOT.name
    export_cache_root.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(CACHE_ROOT, export_cache_root)
else:
    print(f"Skipping cache export because {CACHE_ROOT} does not exist")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
subprocess.run(["zip", "-r", str(ZIP_PATH), str(EXPORT_ROOT)], check=True)
print(f"Wrote {ZIP_PATH}")
print("Upload the export directory or zip as a private Kaggle Dataset to resume a later session.")
